In [5]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
NIFTI_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test/imagesTs"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results_test_with_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check_nnUNet_results_test_with_rCMBs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw > 0 else None
    except:
        return None

# ========================= PROCESAR =========================
df = pd.read_excel(EXCEL_PATH)

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'):
        continue

    print(f"\n🔍 Buscando: {nifti_name}")

    # *** IMAGEN: NIFTI_DIR + nombre_0000.nii.gz ***
    base_name = nifti_name.replace('.nii.gz', '')
    img_name = f"{base_name}_0000.nii.gz"
    img_path = os.path.join(NIFTI_DIR, img_name)
    
    # *** MÁSCARA: MASK_DIR + nombre.nii.gz ***
    mask_name = nifti_name  # nombre original del Excel
    mask_path = os.path.join(MASK_DIR, mask_name)
    
    print(f"  Imagen:  {img_path}")
    print(f"  Máscara: {mask_path}")

    if not os.path.exists(img_path):
        print(f"  Imagen no encontrada")
        continue
    if not os.path.exists(mask_path):
        print(f"  Máscara no encontrada")
        continue

    # Cargar
    img_data = nib.load(img_path).get_fdata()
    mask_data = nib.load(mask_path).get_fdata()
    
    if img_data.shape != mask_data.shape:
        print(f"  Shape mismatch: img {img_data.shape} vs mask {mask_data.shape}")
        continue
        
    shape = img_data.shape
    print(f"  Shape: {shape}")

    # Coordenadas rCMB corregidas (0-based)
    coords = []
    for i in range(1, len(row), 3):
        if i+2 < len(row):
            x = safe_int_minus1(row.iloc[i])
            y = safe_int_minus1(row.iloc[i+1])
            z = safe_int_minus1(row.iloc[i+2])
            if x is not None and y is not None and z is not None:
                if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                    coords.append((x, y, z))

    print(f"  {len(coords)} rCMBs encontrados")

    # *** VISUALIZACIÓN ***
    for cmb_idx, (x, y, z) in enumerate(coords, 1):
        slice_img = img_data[:, :, z]
        slice_mask = mask_data[:, :, z]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
        
        # IZQUIERDA: CÍRCULO rCMB
        vmin = slice_img.mean() - slice_img.std()
        vmax = slice_img.mean() + 2*slice_img.std()
        ax1.imshow(slice_img.T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        circle = plt.Circle((x, y), 5, color='red', fill=False, linewidth=4)
        ax1.add_patch(circle)
        ax1.set_title(f'rCMB #{cmb_idx}\n(x={x},y={y},z={z})')
        ax1.axis('off')
        
        # DERECHA: MÁSCARA ROJA
        ax2.imshow(slice_img.T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        mask_binary = (slice_mask > 0).astype(float)
        ax2.imshow(mask_binary.T, cmap='Reds', alpha=0.7, origin='lower')
        ax2.set_title(f'Máscara sCMB segmentada')
        ax2.axis('off')
        
        plt.suptitle(f"{base_name}")
        out_name = f"{base_name}_CMB{cmb_idx}_validation.png"
        plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=200)
        plt.close(fig)
        
        print(f"  CMB{cmb_idx}: ({x},{y},{z})")

print(f"\nImágenes guardadas en: {OUTPUT_DIR}")



🔍 Buscando: 122_T1_MRI_SWI_BFC_50mm_HM.nii.gz
  Imagen:  /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test/imagesTs/122_T1_MRI_SWI_BFC_50mm_HM_0000.nii.gz
  Máscara: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results_test_with_rCMBs/122_T1_MRI_SWI_BFC_50mm_HM.nii.gz
  Shape: (176, 256, 80)
  2 rCMBs encontrados
  CMB1: (63,174,11)
  CMB2: (124,133,43)

🔍 Buscando: 122_T2_MRI_SWI_BFC_50mm_HM.nii.gz
  Imagen:  /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test/imagesTs/122_T2_MRI_SWI_BFC_50mm_HM_0000.nii.gz
  Máscara: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results_test_with_rCMBs/122_T2_MRI_SWI_BFC_50mm_HM.nii.gz
  Shape: (176, 256, 80)
  5 rCMBs encontrados
  CMB1: (50,166,25)
  CMB2: (64,170,12)
  CMB3: (131,101,38)
  CMB4: (130,122,42)
  CMB5: (125,128,44)

🔍 Buscando: 218_T0_MRI_SWI_BFC_50mm_HM.nii.gz
  Imagen:  /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test

In [7]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
NIFTI_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test/imagesTs"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results_test_with_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check_nnUNet_results_test_with_rCMBs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw > 0 else None
    except:
        return None

# ========================= ANÁLISIS TP/FN =========================
df = pd.read_excel(EXCEL_PATH)
results = []
tp_count = 0
fn_count = 0

print("🔬 ANÁLISIS TP/FN...\n")

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'):
        continue

    print(f"\n📋 {nifti_name}")

    # *** ARCHIVOS ***
    base_name = nifti_name.replace('.nii.gz', '')
    img_name = f"{base_name}_0000.nii.gz"
    mask_name = nifti_name
    
    img_path = os.path.join(NIFTI_DIR, img_name)
    mask_path = os.path.join(MASK_DIR, mask_name)
    
    if not os.path.exists(img_path) or not os.path.exists(mask_path):
        print(f"  ❌ Archivos no encontrados")
        print(f"    Img:  {img_path}")
        print(f"    Mask: {mask_path}")
        continue

    # Cargar datos
    img_data = nib.load(img_path).get_fdata()
    mask_data = nib.load(mask_path).get_fdata()
    
    if img_data.shape != mask_data.shape:
        print(f"  ❌ Shape mismatch")
        continue
        
    shape = img_data.shape
    print(f"  ✅ Shape: {shape}")

    # *** rCMBs DEL EXCEL (ground truth) ***
    rcmb_coords = []
    for i in range(1, len(row), 3):
        if i+2 < len(row):
            x = safe_int_minus1(row.iloc[i])
            y = safe_int_minus1(row.iloc[i+1])
            z = safe_int_minus1(row.iloc[i+2])
            if x is not None and y is not None and z is not None:
                if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                    rcmb_coords.append((x, y, z))

    print(f"  {len(rcmb_coords)} rCMBs encontrados")

    # *** VALIDAR CADA rCMB (TP/FN) ***
    for cmb_idx, (x, y, z) in enumerate(rcmb_coords, 1):
        mask_value = mask_data[x, y, z]
        
        # TP: centroide rCMB está marcado en máscara
        if mask_value > 0.5:
            tp_count += 1
            status = "✅ TP"
        else:
            fn_count += 1
            status = "❌ FN"
            
        results.append({
            'nifti': nifti_name, 
            'cmb_idx': cmb_idx, 
            'x': x, 'y': y, 'z': z,
            'mask_value': mask_value, 
            'status': status
        })
        
        print(f"    rCMB{cmb_idx}: ({x},{y},{z}) {status} (mask={mask_value:.3f})")

    # *** VISUALIZACIÓN ***
    for cmb_idx, (x, y, z) in enumerate(rcmb_coords, 1):
        slice_img = img_data[:, :, z]
        slice_mask = mask_data[:, :, z]
        mask_value = mask_data[x, y, z]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
        
        # IZQUIERDA: CÍRCULO rCMB
        vmin = slice_img.mean() - slice_img.std()
        vmax = slice_img.mean() + 2*slice_img.std()
        ax1.imshow(slice_img.T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        circle = plt.Circle((x, y), 5, color='red', fill=False, linewidth=4)
        ax1.add_patch(circle)
        status_icon = "✅" if mask_value > 0.5 else "❌"
        ax1.set_title(f'{status_icon} rCMB #{cmb_idx}\n(x={x},y={y},z={z})')
        ax1.axis('off')
        
        # DERECHA: MÁSCARA ROJA
        ax2.imshow(slice_img.T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        mask_binary = (slice_mask > 0).astype(float)
        ax2.imshow(mask_binary.T, cmap='Reds', alpha=0.7, origin='lower')
        ax2.set_title(f'Máscara sCMB\nValor centroide: {mask_value:.3f}')
        ax2.axis('off')
        
        plt.suptitle(f"{base_name}", fontsize=14)
        out_name = f"{base_name}_CMB{cmb_idx}_validation.png"
        plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=200)
        plt.close(fig)

# ========================= RESUMEN FINAL =========================
total_rcmbs = len(results)
recall = tp_count / total_rcmbs * 100 if total_rcmbs > 0 else 0

print(f"\n🎯 RESULTADOS FINALES:")
print(f"  rCMBs totales (ground truth): {total_rcmbs}")
print(f"  TP (detectados correctamente): {tp_count}")
print(f"  FN (falsos negativos):        {fn_count}")
print(f"  Recall:                       {recall:.1f}%")
print(f"\n📊 Detalles guardados en: {OUTPUT_DIR}/metricas_tp_fn.csv")

# Guardar CSV
pd.DataFrame(results).to_csv(os.path.join(OUTPUT_DIR, 'metricas_tp_fn.csv'), index=False)


🔬 ANÁLISIS TP/FN...


📋 122_T1_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  2 rCMBs encontrados
    rCMB1: (63,174,11) ✅ TP (mask=1.000)
    rCMB2: (124,133,43) ❌ FN (mask=0.000)


/tmp/ipykernel_58794/38671332.py:126: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=200)
/tmp/ipykernel_58794/38671332.py:126: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=200)



📋 122_T2_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  5 rCMBs encontrados
    rCMB1: (50,166,25) ✅ TP (mask=1.000)
    rCMB2: (64,170,12) ✅ TP (mask=1.000)
    rCMB3: (131,101,38) ❌ FN (mask=0.000)
    rCMB4: (130,122,42) ✅ TP (mask=1.000)
    rCMB5: (125,128,44) ❌ FN (mask=0.000)

📋 218_T0_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  2 rCMBs encontrados
    rCMB1: (73,174,12) ❌ FN (mask=0.000)
    rCMB2: (142,149,55) ❌ FN (mask=0.000)

📋 218_T2_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  1 rCMBs encontrados
    rCMB1: (74,176,13) ❌ FN (mask=0.000)

📋 218_T3_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  1 rCMBs encontrados
    rCMB1: (70,177,15) ✅ TP (mask=1.000)

📋 219_T0_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  1 rCMBs encontrados
    rCMB1: (118,111,41) ❌ FN (mask=0.000)

📋 221_T2_MRI_SWI_BFC_50mm_HM.nii.gz
  ✅ Shape: (176, 256, 80)
  1 rCMBs encontrados
    rCMB1: (136,142,21) ❌ FN (mask=0.000)

📋 222_T0_MRI_SWI_BFC_50mm_HM.nii